# Building Energy System

Optimal combination of segments and periods for building energy supply systems.

Author: Leander Kotzur

Import pandas and the relevant time series aggregation class

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig, SegmentConfig
from tsam.tuning import find_pareto_front

pio.renderers.default = "notebook_connected"
import warnings

# Added to every example notebook: silence the v3 column-order
# FutureWarning in the rendered docs (tsam v4 returns result columns in
# input order; see migration guide).
warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*sorted alphabetically.*"
)

### Input data 

Read in time series from testdata.csv with pandas

In [2]:
raw = pd.read_csv("testdata.csv", index_col=0)
raw = raw.rename(
    columns={
        "T": "Temperature [°C]",
        "Load": "Load [kW]",
        "Wind": "Wind [m/s]",
        "GHI": "Solar [W/m²]",
    }
)

In [3]:
raw = raw.drop(
    columns=[
        "Wind [m/s]",
    ],
)

Use tsam's built-in heatmap plotting for visual comparison of the time series

Plot an example series - in this case the temperature

In [4]:
# Original data heatmaps using tsam.unstack_to_periods() with plotly
unstacked = tsam.unstack_to_periods(raw, period_duration=24)
for col in raw.columns:
    px.imshow(
        unstacked[col].values.T,
        labels={"x": "Day", "y": "Hour", "color": col},
        title=f"Original {col}",
        aspect="auto",
    ).show()

### Tune a hierarchical aggregation with segments in combination with distribution representation

Use the new `find_pareto_front()` function to explore the Pareto-optimal combinations.

In [5]:
pareto_results = find_pareto_front(
    raw,
    period_duration=24,
    max_timesteps=100,
    cluster=ClusterConfig(
        method="hierarchical",
        representation="distribution",
    ),
    n_jobs=-1,
)

Building Pareto front:   0%|          | 0/100 [00:00<?, ?it/s]

Building Pareto front:   2%|▏         | 2/100 [00:03<02:37,  1.61s/it]

Building Pareto front:   3%|▎         | 3/100 [00:05<03:17,  2.04s/it]

Building Pareto front:   6%|▌         | 6/100 [00:08<01:56,  1.24s/it]

Building Pareto front:   9%|▉         | 9/100 [00:09<01:21,  1.12it/s]

...

Building Pareto front:  75%|███████▌  | 75/100 [00:32<00:04,  5.37it/s]

Building Pareto front:  80%|████████  | 80/100 [00:33<00:03,  5.20it/s]

Building Pareto front:  85%|████████▌ | 85/100 [00:34<00:02,  5.07it/s]

Building Pareto front:  90%|█████████ | 90/100 [00:36<00:02,  4.88it/s]

Building Pareto front: 100%|██████████| 100/100 [00:36<00:00,  2.78it/s]

And determine the pareto optimal aggregation up to 100 total time steps. This may take some time...

In [6]:
# Show the last result in the Pareto front
last_result = pareto_results[-1]
print(
    f"Final configuration: {last_result.n_clusters} periods, {last_result.n_segments} segments"
)

Final configuration: 20 periods, 5 segments


And show the results for the last aggregation

In [7]:
# Reconstruct the data from the last Pareto result
reconstructed = last_result.reconstructed

In [8]:
# Reconstructed data heatmaps from last tuned aggregation
unstacked_recon = tsam.unstack_to_periods(reconstructed, period_duration=24)
for col in reconstructed.columns:
    px.imshow(
        unstacked_recon[col].values.T,
        labels={"x": "Day", "y": "Hour", "color": col},
        title=f"Reconstructed {col}",
        aspect="auto",
    ).show()

In [9]:
last_result.n_segments

5

In [10]:
last_result.n_clusters

20

In [11]:
# Example with specific configuration using distribution_minmax representation
result = tsam.aggregate(
    raw,
    n_clusters=14,
    period_duration=24,
    cluster=ClusterConfig(
        method="hierarchical",
        representation="distribution_minmax",
    ),
    segments=SegmentConfig(n_segments=8),
    preserve_column_means=False,
)

In [12]:
# Reconstructed data heatmaps with 8 segments and 14 periods
recon = result.reconstructed
unstacked_recon2 = tsam.unstack_to_periods(recon, period_duration=24)
for col in recon.columns:
    px.imshow(
        unstacked_recon2[col].values.T,
        labels={"x": "Day", "y": "Hour", "color": col},
        title=f"Reconstructed {col} (8 seg, 14 periods)",
        aspect="auto",
    ).show()